In [ ]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

from students.staritsyn.lessons2 import Exercise

X, y = load_iris(return_X_y=True)

X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.25, random_state=42, stratify=y_train_val
)

y_train = (y_train == 0).astype(int)
y_train_val = (y_train_val == 0).astype(int)
y_val = (y_val == 0).astype(int)
y_test = (y_test == 0).astype(int)

num_features = X.shape[1]
n_iter = 25

best_auroc = -1.0
best_params: dict[str, int | float] | None = None


def reset_model():
    model = Exercise.create_logistic_model(num_features, np.random.default_rng(42))
    rng = np.random.default_rng(42)
    model.weights = rng.random(num_features)
    model.bias = np.array(rng.random())
    return model


for lr in np.arange(0.0001, 0.01, 0.0001):
    for batch_size in range(1, 15):
        model = reset_model()

        Exercise.fit(model, X_train, y_train, float(lr), n_iter, int(batch_size))

        auroc = model.metric(X_val, y_val, "AUROC")

        if auroc > best_auroc:
            best_auroc = auroc
            best_params = {
                "lr": float(lr),
                "batch_size": int(batch_size),
            }

print("Best params:", best_params)
print("Best validation AUROC:", best_auroc)

if best_params is None:
    raise ValueError("Не удалось подобрать гиперпараметры")

best_lr = float(best_params["lr"])
best_batch_size = int(best_params["batch_size"])

final_model = reset_model()
Exercise.fit(final_model, X_train_val, y_train_val, best_lr, n_iter, best_batch_size)

test_auroc = final_model.metric(X_test, y_test, "AUROC")

print("Final params:")
print("lr =", best_lr)
print("batch_size =", best_batch_size)
print("Test AUROC =", test_auroc)